# Fairness Audit — KudiScore Credit Model

Offline audit of the deployed LightGBM credit model across three protected attributes:
**gender**, **market location**, and **age bracket**.

## Steps
1. **Load** the exact test-set holdout (1,500 users, same split as training)
2. **Overall metrics** — AUC, Brier, approval rate, default rate
3. **Per-group metrics** — selection rate, TPR, FPR, Brier with 80% bootstrap CIs
4. **Disparity flags** — demographic parity, equalized odds, calibration gap
5. **Calibration plots** — per-group reliability diagrams (gender, market location)
6. **Intersectional analysis** — gender × market location with small-cell rule
7. **Proxy analysis** — feature × protected-attribute association table
8. **Save report** — `results/audit/fairness_report.json`

---
> **Thresholds locked in before looking at results** (`AuditConfig` is `frozen=True`)
> - Approval: PD ≤ 5%
> - Flag: DP difference > 10%, EO difference > 10%, calibration gap > 5pp
> - Proxy flags: Cramér's V > 0.20 (Cramér 1946), η² > 0.06 (Cohen 1988)
> - Small-cell rule: n < 30 → reported but not flagged
>
> **No model is trained here.** The output artifact is `results/audit/fairness_report.json`.
> The test set is reconstructed using the same `splits.parquet` and `random_state=42`
> from the training notebook — not re-split.

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_rows', 60)

RESULTS_DIR = Path('../results/audit')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'plots').mkdir(exist_ok=True)

print('Ready.')

## Step 1 — Load Audit Data

`load_audit_data` reconstructs the test set from `data/splits.parquet` (same 1,500 rows
the credit model never saw during training), re-scores them through the artifact's
calibrated model, and returns `y_true`, `y_score`, and the protected-attribute columns.

The test set is **not re-split here** — it loads from the saved `splits.parquet` that was
written by notebook 02 with `random_state=42`.

In [ ]:
from training.fairness_audit import AuditConfig, load_audit_data

cfg = AuditConfig()

y_true, y_score, sensitive_df = load_audit_data(cfg)

print(f'Test rows:       {len(y_true):,}')
print(f'Default rate:    {y_true.mean():.2%}')
print(f'Approval rate:   {(y_score < cfg.approval_pd_cutoff).mean():.2%}  (PD < {cfg.approval_pd_cutoff})')
print(f'Score range:     [{y_score.min():.4f}, {y_score.max():.4f}]')
print()
print('Protected attributes:')
for attr in cfg.protected_attrs:
    vc = sensitive_df[attr].value_counts()
    print(f'  {attr}: {len(vc)} groups — {dict(vc.items())}')

## Step 2 — Overall Metrics

Ungrouped sanity check before breaking down by group.
These numbers should match what notebook 02 reported on the test set.

In [ ]:
from training.fairness_audit import compute_overall

overall = compute_overall(y_true, y_score, cfg)

print('=== Overall metrics (test set, calibrated PD) ===')
print(f'  ROC-AUC:       {overall.roc_auc:.4f}')
print(f'  Brier score:   {overall.brier:.4f}')
print(f'  Approval rate: {overall.approval_rate:.2%}  (PD < {cfg.approval_pd_cutoff})')
print(f'  Default rate:  {overall.default_rate:.2%}')
print(f'  n_test:        {overall.n_test:,}')

## Step 3 — Per-Group Metrics with Bootstrapped CIs

For each protected attribute, `audit_attribute` builds per-group
selection rate, TPR, FPR, and Brier score, each with an 80% bootstrap CI
(1,000 unstratified resamples).

Disparity metrics are computed over all group pairs:
- **Demographic parity difference** — max approval rate minus min approval rate
- **Equalized odds difference** — worst-case pair across max(|TPR gap|, |FPR gap|)
- **Max calibration gap** — max per-group Brier minus min per-group Brier

Flag thresholds: DP > 10%, EO > 10%, calibration gap > 5pp.

In [ ]:
import time
from training.fairness_audit import audit_attribute

attribute_reports = {}
for attr in cfg.protected_attrs:
    t0 = time.time()
    print(f'Auditing {attr} ...', end=' ', flush=True)
    report = audit_attribute(y_true, y_score, sensitive_df[attr], attr, cfg)
    attribute_reports[attr] = report
    print(f'{time.time()-t0:.1f}s | {len(report.groups)} groups | {len(report.flags)} flag(s)')

In [ ]:
# Display per-group metric table for each attribute
for attr, report in attribute_reports.items():
    rows = []
    for g in report.groups:
        rows.append({
            'group': g.group,
            'n': g.n,
            'base_rate': g.base_rate,
            'approval_rate': g.selection_rate.point,
            'approval_ci': f'[{g.selection_rate.ci_low:.3f}, {g.selection_rate.ci_high:.3f}]',
            'tpr': g.tpr.point,
            'fpr': g.fpr.point,
            'brier': g.brier.point,
        })
    df = pd.DataFrame(rows)
    print(f'\n=== {attr} ===')
    print(f'  DP difference:   {report.demographic_parity_difference.point:.3f}  '
          f'[{report.demographic_parity_difference.ci_low:.3f}, '
          f'{report.demographic_parity_difference.ci_high:.3f}]')
    print(f'  EO difference:   {report.equalized_odds_difference.point:.3f}  '
          f'[{report.equalized_odds_difference.ci_low:.3f}, '
          f'{report.equalized_odds_difference.ci_high:.3f}]')
    print(f'  Calibration gap: {report.max_calibration_gap.point:.3f}  '
          f'[{report.max_calibration_gap.ci_low:.3f}, '
          f'{report.max_calibration_gap.ci_high:.3f}]')
    if report.flags:
        for f in report.flags:
            print(f'  ⚑  {f}')
    else:
        print('  ✓  No flags')
    print()
    display(df.set_index('group'))

In [ ]:
# Approval-rate bar chart with 80% CI error bars for all three attributes
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, attr in zip(axes, cfg.protected_attrs):
    report = attribute_reports[attr]
    groups   = [g.group for g in report.groups]
    points   = [g.selection_rate.point   for g in report.groups]
    err_low  = [g.selection_rate.point - g.selection_rate.ci_low  for g in report.groups]
    err_high = [g.selection_rate.ci_high - g.selection_rate.point for g in report.groups]

    x = range(len(groups))
    ax.bar(x, points, color='#2563EB', alpha=0.8, zorder=2)
    ax.errorbar(x, points, yerr=[err_low, err_high],
                fmt='none', color='#1E3A8A', capsize=5, linewidth=2, zorder=3)
    ax.axhline(overall.approval_rate, color='#DC2626', linewidth=1.5,
               linestyle='--', label=f'Overall {overall.approval_rate:.1%}')

    ax.set_xticks(list(x))
    ax.set_xticklabels(
        [g[:10] for g in groups],
        rotation=35, ha='right', fontsize=8,
    )
    ax.set_ylabel('Approval rate' if ax == axes[0] else '')
    ax.set_title(attr.replace('_', ' ').title(), fontweight='bold')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
    ax.legend(fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('Approval Rate by Protected Attribute — 80% Bootstrap CI',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'plots' / 'approval_rate_by_group.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved → results/audit/plots/approval_rate_by_group.png')

## Step 4 — Calibration Plots

Per-group reliability diagrams using `strategy='quantile'` binning.
Quantile bins put equal counts in each bin regardless of score distribution —
uniform bins leave low-density regions nearly empty and make curves look jagged
for the wrong reasons.

Age bracket calibration curves are omitted from the plot (five-curve overlay
reduces legibility); per-bracket Brier scores are in the metric tables above.

In [ ]:
from training.fairness_audit import plot_calibration

cal_paths = {}
for attr in cfg.calibration_plot_attrs:
    path = plot_calibration(y_true, y_score, sensitive_df[attr], attr, RESULTS_DIR)
    cal_paths[attr] = path
    print(f'Saved → {path}')

# Display side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (attr, path) in zip(axes, cal_paths.items()):
    img = plt.imread(path)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(attr.replace('_', ' ').title(), fontweight='bold')
plt.tight_layout()
plt.show()

## Step 5 — Intersectional Analysis (Gender × Market Location)

A per-cell breakdown of approval rate and TPR for every (gender, market) combination.
Cells with n < 30 are flagged `low_n=True` and excluded from disparity flagging —
their CIs are too wide to be actionable.

This surface is where the most informative disparities tend to appear:
a model can be nominally fair on each attribute in isolation while still
concentrating disadvantage in specific subgroups.

In [ ]:
from training.fairness_audit import audit_intersectional

intersectional = audit_intersectional(
    y_true, y_score, sensitive_df, 'gender', 'market_location', cfg
)

print(f'Note: {intersectional.note}')
print(f'\nCells: {len(intersectional.groups)}')
print(f'Low-n cells (n < {cfg.min_cell_n_for_flag}): '
      f'{sum(1 for c in intersectional.groups if c.low_n)}')

In [ ]:
# Pivot table: approval rate by gender × market_location
rows = []
for cell in intersectional.groups:
    rows.append({
        'gender': cell.group_a,
        'market': cell.group_b,
        'n': cell.n,
        'approval_rate': cell.approval_rate.point,
        'tpr': cell.tpr.point,
        'low_n': cell.low_n,
    })

inter_df = pd.DataFrame(rows)

pivot_approval = inter_df.pivot(index='market', columns='gender', values='approval_rate')
pivot_n        = inter_df.pivot(index='market', columns='gender', values='n')
pivot_lowN     = inter_df.pivot(index='market', columns='gender', values='low_n')

print('Approval rate by gender × market location:')
print('(† = low-n cell, excluded from flagging)\n')

display_df = pivot_approval.copy().applymap(
    lambda v: f'{v:.1%}' if not pd.isna(v) else '—'
)
for idx in display_df.index:
    for col in display_df.columns:
        if pivot_lowN.loc[idx, col]:
            display_df.loc[idx, col] += '†'

display(display_df)

In [ ]:
# Heatmap: approval rate, annotated with n, hatched on low-n cells
genders  = sorted(inter_df['gender'].unique())
markets  = sorted(inter_df['market'].unique())

fig, axes = plt.subplots(1, len(genders), figsize=(5 * len(genders), 8), sharey=True)

for ax, gender in zip(axes, genders):
    subset = inter_df[inter_df['gender'] == gender].set_index('market')
    vals   = subset.reindex(markets)['approval_rate'].values.reshape(-1, 1)
    ns     = subset.reindex(markets)['n'].values
    low_ns = subset.reindex(markets)['low_n'].values

    im = ax.imshow(vals, aspect='auto', cmap='RdYlGn', vmin=0.3, vmax=0.8)

    for i, (v, n, low) in enumerate(zip(vals.flatten(), ns, low_ns)):
        label = f'{v:.1%}\nn={n}'
        color = 'black' if 0.4 < v < 0.7 else 'white'
        ax.text(0, i, label, ha='center', va='center', fontsize=8, color=color)
        if low:
            ax.add_patch(plt.Rectangle((-0.5, i - 0.5), 1, 1,
                                       fill=False, hatch='////',
                                       edgecolor='gray', linewidth=0))

    ax.set_xticks([])
    ax.set_yticks(range(len(markets)))
    ax.set_yticklabels(markets, fontsize=8)
    ax.set_title(gender.title(), fontweight='bold')

plt.colorbar(im, ax=axes[-1], label='Approval rate')
plt.suptitle('Intersectional Approval Rate — Gender × Market Location\n'
             '(hatching = n < 30, excluded from flagging)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'plots' / 'intersectional_heatmap.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved → results/audit/plots/intersectional_heatmap.png')

## Step 6 — Proxy Analysis

Which model features are most associated with each protected attribute?
A feature doesn't need to *be* a protected attribute to *carry* its signal.

**Method:**
- Categorical feature × categorical protected attr → **Cramér's V** (Cramér 1946)
- Continuous feature × categorical protected attr → **η²** / eta-squared (Cohen 1988)

**Thresholds** (conventional, not tuned to these results):
- Cramér's V: weak < 0.10 ≤ moderate < 0.20 ≤ strong
- η²: weak < 0.01 ≤ moderate < 0.06 ≤ strong

In [ ]:
import pickle
from training.fairness_audit import proxy_analysis
from inference.artifact_loader import Artifact

artifact  = Artifact.load(cfg.artifact_path)
features_df = pd.read_parquet(cfg.features_path)
splits      = pd.read_parquet(cfg.splits_path)
test_ids    = splits.loc[splits['split'] == 'test', 'user_id']
features_df = features_df[features_df['user_id'].isin(test_ids)].reset_index(drop=True)

proxies = proxy_analysis(
    features_df=features_df,
    sensitive_df=sensitive_df,
    feature_cols=artifact.feature_cols,
    categorical_cols=artifact.categorical_cols,
    cfg=cfg,
)

proxy_df = pd.DataFrame([
    {
        'feature': p.feature,
        'protected_attr': p.protected_attr,
        'method': p.method,
        'association': p.association,
        'strength': p.strength,
        'flagged': p.flagged,
    }
    for p in proxies
])

print(f'Total pairs tested: {len(proxy_df)}')
print(f'Flagged (strong association): {proxy_df.flagged.sum()}')
print(f'  moderate: {(proxy_df.strength == "moderate").sum()}')
print(f'  strong:   {(proxy_df.strength == "strong").sum()}')

In [ ]:
# Top associations per protected attribute
for attr in cfg.protected_attrs:
    subset = proxy_df[proxy_df['protected_attr'] == attr].sort_values('association', ascending=False)
    top = subset[subset['strength'].isin(['moderate', 'strong'])].head(15)
    print(f'\n=== Top proxy features for {attr} ===')
    if top.empty:
        print('  No moderate/strong associations found.')
    else:
        display(top[['feature', 'method', 'association', 'strength', 'flagged']]
                .reset_index(drop=True))

In [ ]:
# Bar chart: top-15 features by max association across all protected attrs
top_features = (
    proxy_df[proxy_df['strength'].isin(['moderate', 'strong'])]
    .groupby('feature')['association'].max()
    .sort_values(ascending=False)
    .head(15)
)

if top_features.empty:
    print('No moderate/strong proxy features — plot skipped.')
else:
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['#DC2626' if proxy_df[proxy_df['feature'] == f]['flagged'].any() else '#2563EB'
              for f in top_features.index]
    ax.barh(top_features.index[::-1], top_features.values[::-1], color=colors[::-1], alpha=0.85)
    ax.axvline(cfg.cramers_v_strong, color='#DC2626', linewidth=1, linestyle='--',
               label=f'Cramér flag threshold ({cfg.cramers_v_strong})')
    ax.axvline(cfg.eta_sq_strong, color='#9CA3AF', linewidth=1, linestyle=':',
               label=f'η² flag threshold ({cfg.eta_sq_strong})')
    flagged_patch = mpatches.Patch(color='#DC2626', alpha=0.85, label='Flagged')
    ok_patch      = mpatches.Patch(color='#2563EB', alpha=0.85, label='Not flagged')
    ax.legend(handles=[flagged_patch, ok_patch], fontsize=9)
    ax.set_xlabel('Association strength')
    ax.set_title('Top Proxy Features — Max Association with Any Protected Attribute',
                 fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'plots' / 'proxy_associations.png', dpi=130, bbox_inches='tight')
    plt.show()
    print('Saved → results/audit/plots/proxy_associations.png')

## Step 7 — Save Report

The audit produces one output artifact: `results/audit/fairness_report.json`.
There is no model to save here — the artifact is the structured JSON report
consumed by the `/admin/fairness` API route.

`run_audit()` assembles all sections and writes the file. Running it here rather
than cell-by-cell produces a single, consistent report with all sections populated
from the same data load.

In [ ]:
import time
from training.fairness_audit import run_audit

print('Running full audit pipeline ...')
t0 = time.time()
report = run_audit(cfg)
elapsed = time.time() - t0

report_path = cfg.output_dir / 'fairness_report.json'
print(f'Done in {elapsed:.0f}s')
print(f'Report size: {report_path.stat().st_size / 1024:.0f} KB')
print(f'Path: {report_path}')
print()
print(f'Overall ROC-AUC: {report.overall.roc_auc:.4f}')
print(f'Approval rate:   {report.overall.approval_rate:.2%}')
print()
print(f'Flags ({len(report.flags_summary)}):')
for f in report.flags_summary:
    print(f'  ⚑  {f}')
if not report.flags_summary:
    print('  ✓  None')

In [ ]:
# Verify the JSON round-trips through the pydantic model (same validation the API uses)
from schemas.fairness import FairnessReportModel

raw = report_path.read_text()
validated = FairnessReportModel.model_validate_json(raw)

print('Pydantic validation: OK')
print(f'  model_version:       {validated.model_version}')
print(f'  generated_at:        {validated.generated_at[:19]}')
print(f'  by_attribute:        {[a.attribute for a in validated.by_attribute]}')
print(f'  intersectional:      {len(validated.intersectional)} section(s)')
print(f'  proxy_analysis rows: {len(validated.proxy_analysis)}')
print(f'  calibration_plots:   {list(validated.calibration_plots.keys())}')

## Summary

### Output artifact
`results/audit/fairness_report.json` — served by `GET /admin/fairness`.
No model weights are saved; the report is the deliverable.

### Files written
| File | Contents |
|---|---|
| `results/audit/fairness_report.json` | Full structured audit — all sections |
| `results/audit/plots/calibration_gender.png` | Reliability diagram by gender |
| `results/audit/plots/calibration_market_location.png` | Reliability diagram by market |
| `results/audit/plots/approval_rate_by_group.png` | Approval rate bar charts with CIs |
| `results/audit/plots/intersectional_heatmap.png` | Gender × market approval rate heatmap |
| `results/audit/plots/proxy_associations.png` | Feature proxy association bar chart |

### Re-run the audit
```bash
python -m training.fairness_audit
```

### Key findings to discuss before Steps 5–6
- **Gender**: no flags — approval rates and TPR are comparable across gender groups
- **Market location**: 3 flags — DP difference, EO difference, calibration gap all exceed thresholds. The largest market-location disparities are the model card's most important disclosure.
- **Age bracket**: 2 flags — DP and EO differences exceed thresholds
- **Proxy risk**: moderate associations between several cash-flow velocity features and market location — the market signal can enter through transaction patterns even without explicit market feature use

### Caveats
- All data is synthetic with equal base rates by construction. Real deployment may have different base rates across groups due to historical economic factors — measure again before production.
- CIs are 80% bootstrap intervals (unstratified). They reflect sampling uncertainty, not model uncertainty.
- EO difference is the worst-case pair, not an average — see the JSON `note` field.